<a href="http://landlab.github.io"><img style="float: left; width: 300px;" src="https://landlab.csdms.io/_static/landlab_logo.png"></a>

# 2D Surface Water Flow component — Subcritical Uniform Flow (HLLC)

<hr>
<small>For more Landlab tutorials, click here: <a href="https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html">https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html</a></small>
<hr>


## Overview

This tutorial demonstrates **subcritical uniform flow** in a rectangular channel and validates the `RiverFlowDynamics_HLLC` component against the **Manning analytical solution**.

The setup is identical to Tutorial 3 for `RiverFlowDynamics`: a mild slope (S = 0.001), natural-channel roughness (n = 0.025), and an inlet prescribed at the analytical normal depth. Running the same case with both solvers allows a direct comparison.

> **Key numerical differences from `RiverFlowDynamics`:**
>
> | Feature | RiverFlowDynamics | RiverFlowDynamics_HLLC |
> |---|---|---|
> | Scheme | Semi-implicit, semi-Lagrangian | Explicit Godunov-type (HLLC flux) |
> | Time stepping | Fixed user-supplied dt | Adaptive CFL (automatic) |
> | Shock capturing | No | Yes |
> | Velocity storage | Scalar at links | x/y components at nodes |
> | Subcritical inlet BC | Fixes h and u at entry links | Fixes h and u at entry nodes |

### Analytical background

For a wide rectangular channel the Manning equation simplifies to:

$$u_n = \frac{1}{n} h_n^{2/3} S^{1/2}$$

which can be rearranged to find normal depth given the unit discharge $q = h_n u_n$:

$$h_n = \left(\frac{n\,q}{S^{1/2}}\right)^{3/5}$$

For the prescribed conditions (n = 0.025, S = 0.001, h_n = 0.4 m) the flow is **subcritical** (Fr = 0.347), which is the valid regime for both schemes.

The channel is **mild** when $S < S_c$, where $S_c$ is the critical slope ($S_c \approx 0.011$ for these parameters). With $S = 0.001 \ll S_c$, the normal depth is subcritical and the semi-implicit solver is in its valid regime.

### Subcritical inlet BC note for HLLC

For Fr < 1 only **one** characteristic enters the domain from outside. Fixing both h and u at the inlet over-constrains the Riemann problem, which can cause the interior to settle at a slightly different equilibrium than the prescribed normal depth. This is a known property of explicit Godunov-type solvers with Dirichlet inflow BCs for subcritical flow. The tutorial documents and quantifies this effect.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

from landlab import RasterModelGrid
from landlab.components import RiverFlowDynamics_HLLC
from landlab.plot.imshow import imshow_grid

## Information about the component

Using the class name as argument for the `help` function returns descriptions of the various methods and parameters.

In [ ]:
help(RiverFlowDynamics_HLLC)

## Examples

-- --

### Example: Subcritical uniform flow in a 10 m natural channel

A natural channel (gravel bed, Manning's n = 0.025), 10 m long and 1 m wide, with a gentle slope of S = 0.001 m/m. The inflow is prescribed at the analytical normal depth so that the steady-state solution should be uniform flow throughout the channel.

In [ ]:
# Physical parameters
mannings_n = 0.025  # Manning's roughness coefficient [s/m^(1/3)]
channel_slope = 0.001  # Bed slope [m/m] — mild, subcritical

# Analytical normal depth (wide-channel Manning, R = h)
g = 9.81
S = channel_slope
n = mannings_n
h_n = 0.4  # normal depth [m]
u_n = (1 / n) * h_n ** (2 / 3) * S**0.5  # normal velocity [m/s]
q_n = h_n * u_n  # unit discharge [m²/s]
Fr_n = u_n / (g * h_n) ** 0.5  # Froude number
h_c = (q_n**2 / g) ** (1 / 3)  # critical depth [m]
print(f"Normal depth:    h_n = {h_n:.4f} m")
print(f"Normal velocity: u_n = {u_n:.4f} m/s")
print(f"Unit discharge:  q   = {q_n:.4f} m²/s")
print(
    f'Froude number:   Fr  = {Fr_n:.4f}  → {"subcritical ✓" if Fr_n < 1 else "supercritical"}'
)
print(f"Critical depth:  h_c = {h_c:.4f} m")

# Simulation parameters
target_time = 300.0  # total simulated time [s]
display_dt = 10.0  # redraw plot every this many simulated seconds
nrows = 20  # grid rows
ncols = 100  # grid columns
dx = 0.1  # node spacing x [m]
dy = 0.1  # node spacing y [m]

Create the grid:

In [ ]:
grid = RasterModelGrid((nrows, ncols), xy_spacing=(dx, dy))

Create the elevation field. The channel bed slopes gently from left to right (S = 0.001). Bank nodes are raised to act as channel walls:

In [ ]:
te = grid.add_field(
    "topographic__elevation", 1.0 - channel_slope * grid.x_of_node, at="node"
)
te[grid.y_of_node > 1.5] = 2.5  # north bank
te[grid.y_of_node < 0.5] = 2.5  # south bank

Visualise the channel topography:

In [ ]:
nrows, ncols = grid.shape
mid_row = nrows // 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

plt.sca(ax1)
imshow_grid(
    grid, "topographic__elevation", cmap="terrain", colorbar_label="Elevation (m)"
)
ax1.set_title("Channel Topography")

x_profile = grid.x_of_node.reshape((nrows, ncols))[mid_row, :]
z_profile = grid.at_node["topographic__elevation"].reshape((nrows, ncols))[mid_row, :]
fixed_y_min = z_profile.min() - 0.2

ax2.plot(x_profile, z_profile, color="saddlebrown", lw=2, label="Bed Surface")
ax2.fill_between(x_profile, fixed_y_min, z_profile, color="saddlebrown", alpha=0.4)
ax2.set_title("Longitudinal Profile (Centerline)")
ax2.set_xlabel("Distance (m)")
ax2.set_ylabel("Elevation (m)")
ax2.set_ylim(fixed_y_min, z_profile.max() + 0.6)
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

The channel starts empty. Only the **depth field** is required as input —
`surface_water__velocity` (at links) and `surface_water__elevation` are
**created automatically** by the HLLC component.

In [ ]:
# Channel starts dry — water enters from the left inlet and fills the domain
h = grid.add_zeros("surface_water__depth", at="node")

Set the inflow boundary condition. HLLC uses **node-centred velocity components** (u, v) at entry nodes — no link indices needed. We also set `wall_edges` to enforce reflective (zero normal flux) boundaries at the top and bottom of the domain.

In [ ]:
# Left-edge channel nodes (rows 5-15, col 0)
fixed_entry_nodes = np.array(
    [500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500]
)

entry_nodes_h_values = np.full(len(fixed_entry_nodes), h_n)  # depth = h_n
entry_nodes_u_values = np.full(len(fixed_entry_nodes), u_n)  # x-velocity = u_n
entry_nodes_v_values = np.zeros(len(fixed_entry_nodes))  # no cross-flow

Visualise the inflow cross-section:

In [ ]:
y_ground = grid.y_of_node[grid.nodes_at_left_edge]
z_ground = te[grid.nodes_at_left_edge]
constant_wse = (entry_nodes_h_values + te[fixed_entry_nodes])[0]

y_intersections = []
for i in range(len(y_ground) - 1):
    if (z_ground[i] - constant_wse) * (z_ground[i + 1] - constant_wse) <= 0:
        s = (z_ground[i + 1] - z_ground[i]) / (y_ground[i + 1] - y_ground[i])
        y_intersections.append(y_ground[i] + (constant_wse - z_ground[i]) / s)

fig, ax = plt.subplots(figsize=(6.5, 3.9))
ax.fill_between(y_ground, 0.75, z_ground, color="saddlebrown", alpha=0.4, zorder=1)
ax.plot(
    y_ground,
    z_ground,
    color="saddlebrown",
    linewidth=2.5,
    label="Channel Bed",
    zorder=3,
)
if len(y_intersections) >= 2:
    ax.plot(
        [y_intersections[0], y_intersections[-1]],
        [constant_wse, constant_wse],
        color="dodgerblue",
        linewidth=2.5,
        label=f"Water Surface (h_n={h_n} m)",
        zorder=4,
    )
    wse_arr = np.full_like(z_ground, constant_wse)
    ax.fill_between(
        y_ground,
        z_ground,
        wse_arr,
        where=(z_ground <= wse_arr),
        color="deepskyblue",
        alpha=0.5,
        interpolate=True,
        zorder=2,
    )
ax.set_title(
    f"Channel Cross-section at Inlet  " f"(h_n = {h_n:.3f} m, u_n = {u_n:.3f} m/s)",
    fontsize=12,
    fontweight="bold",
)
ax.set_xlabel("Distance across channel [m]", fontsize=11)
ax.set_ylabel("Elevation [m]", fontsize=11)
ax.set_xlim(0.25, 1.75)
ax.set_ylim(0.75, 1.75)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(loc="upper center", framealpha=1.0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

Instantiate `RiverFlowDynamics_HLLC`. Key differences from `RiverFlowDynamics`:

- **No `dt`** at construction — the adaptive CFL timestep is used automatically.
- **`wall_edges={"top", "bottom"}`** enforces reflective walls at channel sides.
- **`entry_nodes_u_values` / `entry_nodes_v_values`** replace `entry_links_vel_values`.

In [ ]:
right_edge = grid.nodes_at_right_edge
channel_exit_mask = (grid.y_of_node[right_edge] >= 0.5) & (
    grid.y_of_node[right_edge] <= 1.5
)
exit_nodes = right_edge[channel_exit_mask]
exit_eta = te[exit_nodes] + h_n  # WSE = bed + normal depth (channel only)

rfd = RiverFlowDynamics_HLLC(
    grid,
    mannings_n=mannings_n,
    fixed_entry_nodes=fixed_entry_nodes,
    entry_nodes_h_values=entry_nodes_h_values,
    entry_nodes_u_values=entry_nodes_u_values,
    entry_nodes_v_values=entry_nodes_v_values,
    fixed_exit_nodes=exit_nodes,
    exit_nodes_eta_values=exit_eta,
    outlet_max_depth=h_n,
    wall_edges={"top", "bottom"},
)

### What result to expect

| Quantity | Value |
|---|---|
| Slope | S = 0.001  (mild, S < S_c ≈ 0.011) |
| Normal depth | h_n = **0.400 m** (Fr = 0.347, subcritical) |
| Critical depth | h_c = 0.197 m |
| Inlet BC | h = h_n = 0.400 m, u = u_n = 0.687 m/s |

Because the flow is **subcritical** (Fr < 1), only one characteristic enters the domain at the inlet. Fixing both h and u over-constrains the Riemann problem — the Godunov solver resolves this by allowing a small flux imbalance at the first interior cell, which may result in a slightly different equilibrium depth than the analytical h_n. The validation cell below quantifies this. For comparison, `RiverFlowDynamics` handles subcritical inlets more naturally via its implicit pressure-correction solve.

In [ ]:
# Pre-calculate geometry and fixed axis limits
nrows, ncols = grid.shape
mid_row = nrows // 2
z_prof_init = grid.at_node["topographic__elevation"].reshape((nrows, ncols))[mid_row, :]
fixed_x_max = np.max(grid.x_of_node)
fixed_y_min = np.min(z_prof_init) - 0.2
fixed_y_max = np.max(z_prof_init) + 0.8


def update_display(t0, show_progress=False):
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    plt.sca(ax1)
    grid.imshow("surface_water__depth", limits=(0.0, 0.6))
    ax1.set_title(
        f"Water Depth (m)  [t = {rfd.elapsed_time:.2f} s / {target_time:.0f} s]"
    )

    x_profile = grid.x_of_node.reshape((nrows, ncols))[mid_row, :]
    z_profile = grid.at_node["topographic__elevation"].reshape((nrows, ncols))[
        mid_row, :
    ]
    wse_profile = grid.at_node["surface_water__elevation"].reshape((nrows, ncols))[
        mid_row, :
    ]
    wse_clean = np.maximum(wse_profile, z_profile)

    ax2.plot(x_profile, z_profile, color="saddlebrown", lw=2, label="Bed Surface")
    ax2.fill_between(x_profile, fixed_y_min, z_profile, color="saddlebrown", alpha=0.4)
    ax2.plot(x_profile, wse_clean, color="deepskyblue", lw=2, label="Water Surface")
    ax2.fill_between(x_profile, z_profile, wse_clean, color="deepskyblue", alpha=0.5)
    ax2.plot(
        x_profile,
        z_profile + h_n,
        color="limegreen",
        lw=2,
        ls="--",
        label=f"Analytical h_n = {h_n:.3f} m  (Fr = {Fr_n:.3f})",
    )
    ax2.set_xlim(0, fixed_x_max)
    ax2.set_ylim(fixed_y_min, fixed_y_max)
    ax2.set_title("Longitudinal Profile (Centerline)")
    ax2.set_xlabel("Distance [m]")
    ax2.set_ylabel("Elevation [m]")
    ax2.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

    if show_progress:
        wall = time.time() - t0
        pct = rfd.elapsed_time / target_time
        eta = wall * (1.0 / pct - 1.0) if pct > 0 else float("nan")
        print(
            f"sim-time {rfd.elapsed_time:.2f} / {target_time:.1f} s  "
            f"({pct:.1%})  wall {wall:.1f} s  ETA {eta:.1f} s  "
            f"dt={rfd.current_dt:.4f} s"
        )


# ── Run the simulation ─────────────────────────────────────────────────────
t0 = time.time()
next_display_t = 0.0

update_display(t0, show_progress=False)

while rfd.elapsed_time < target_time:
    rfd.run_one_step()
    if rfd.elapsed_time >= next_display_t:
        update_display(t0, show_progress=True)
        next_display_t += display_dt

update_display(t0, show_progress=False)  # final frame
print("Done.")

## Validation against the Manning analytical solution

At steady state the simulated depth along the centerline should be close to h_n = 0.400 m. Any residual reflects the subcritical inlet over-constraint discussed above.

In [ ]:
nrows, ncols = grid.shape
mid_row = nrows // 2
x_vals = grid.x_of_node.reshape((nrows, ncols))[mid_row, :]
z_vals = grid.at_node["topographic__elevation"].reshape((nrows, ncols))[mid_row, :]
h_sim = grid.at_node["surface_water__depth"].reshape((nrows, ncols))[mid_row, :]

h_interior = h_sim[1:-1]
mae_cm = np.mean(np.abs(h_interior - h_n)) * 100
max_err_cm = np.max(np.abs(h_interior - h_n)) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(x_vals, h_sim, color="deepskyblue", lw=2, label="Simulated h(x)")
ax1.axhline(
    h_n, color="limegreen", lw=2, ls="--", label=f"Analytical h_n = {h_n:.3f} m"
)
ax1.set_xlabel("Distance [m]")
ax1.set_ylabel("Water depth [m]")
ax1.set_title(f"Depth profile at t = {rfd.elapsed_time:.1f} s  (MAE = {mae_cm:.3f} cm)")
ax1.set_ylim(0, 0.6)
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.5)

ax2.plot(
    x_vals[1:-1],
    (h_interior - h_n) * 100,
    color="tomato",
    lw=1.5,
    label="Error = h_sim − h_n",
)
ax2.axhline(0, color="k", lw=0.8, ls=":")
ax2.set_xlabel("Distance [m]")
ax2.set_ylabel("Depth error [cm]")
ax2.set_title("Pointwise error relative to analytical h_n")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Analytical normal depth:  h_n = {h_n:.4f} m")
print(f"Mean simulated depth:      h̄  = {h_interior.mean():.4f} m")
print(f"Mean absolute error (MAE):     {mae_cm:.4f} cm")
print(f"Max absolute error:            {max_err_cm:.4f} cm")
print(f"Relative MAE:                  {mae_cm / h_n / 100 * 100:.3f} %")

### Velocity analysis

With HLLC, velocity is stored as **x- and y-components at nodes** — no link interpolation required.

In [ ]:
u_arr = grid.at_node["surface_water__x_velocity"].reshape((nrows, ncols))
v_arr = grid.at_node["surface_water__y_velocity"].reshape((nrows, ncols))
vel_mag = np.sqrt(u_arr**2 + v_arr**2)
h_arr = grid.at_node["surface_water__depth"].reshape((nrows, ncols))

u_mid = u_arr[mid_row, 1:-1]
h_mid = h_arr[mid_row, 1:-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(x_vals[1:-1], u_mid, color="steelblue", lw=2, label="Simulated u(x)")
ax1.axhline(
    u_n, color="limegreen", lw=2, ls="--", label=f"Analytical u_n = {u_n:.3f} m/s"
)
ax1.set_ylim(u_n * 0.95, u_n * 1.05)
ax1.set_xlabel("Distance [m]")
ax1.set_ylabel("x-velocity [m/s]")
ax1.set_title("Centerline velocity profile")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.5)

vel_masked = np.ma.masked_where(h_arr < 0.001, vel_mag)
im = ax2.imshow(
    vel_masked,
    origin="lower",
    cmap="plasma",
    extent=[
        grid.x_of_node.min(),
        grid.x_of_node.max(),
        grid.y_of_node.min(),
        grid.y_of_node.max(),
    ],
)
fig.colorbar(im, ax=ax2, label="Velocity magnitude [m/s]")
ax2.set_xlabel("X [m]")
ax2.set_ylabel("Y [m]")
ax2.set_title("Velocity magnitude map")
plt.tight_layout()
plt.show()

n_eff = (1 / u_mid.mean()) * h_mid.mean() ** (2 / 3) * S**0.5
print(f"Mean u (interior) = {u_mid.mean():.4f} m/s  (target {u_n:.4f} m/s)")
print(f"Effective n       = {n_eff:.4f}  (prescribed {n:.3f})")

### Discharge verification

At steady state continuity requires Q = h · u · W = constant along the channel. With HLLC, momentum hu is stored directly in `surface_water__x_momentum` — no link interpolation needed.

In [ ]:
hu = grid.at_node["surface_water__x_momentum"].reshape((nrows, ncols))

# Integrate hu * dy over the 10 channel rows (y = 0.5 to 1.4 m, width = 1.0 m).
row_start = int(0.5 / dy)  # row 5,  y = 0.5 m
row_end = int(1.5 / dy) - 1  # row 14, y = 1.4 m
channel_rows = slice(row_start, row_end + 1)
Q_x = np.sum(hu[channel_rows, :], axis=0) * dy
x_cols = grid.x_of_node.reshape((nrows, ncols))[mid_row, :]

# Exclude boundary columns — inlet (col 0) and outlet (col -1) have BC artifacts
x_int = x_cols[1:-1]
Q_int = Q_x[1:-1]

Q_analytical = h_n * u_n * 1.0  # W = 1 m

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_int, Q_int, color="steelblue", lw=2, label="Simulated Q(x)")
ax.axhline(
    Q_analytical,
    color="limegreen",
    lw=2,
    ls="--",
    label=f"Analytical Q = {Q_analytical:.4f} m³/s",
)
# Explicit y-limits: ±5% around the analytical value so the scale is physical
ax.set_ylim(Q_analytical * 0.95, Q_analytical * 1.05)
ax.set_xlabel("Distance [m]")
ax.set_ylabel("Discharge Q [m³/s]")
ax.set_title("Discharge Along Channel — Continuity Check")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Analytical Q          = {Q_analytical:.4f} m³/s")
print(f"Mean Q (interior)     = {Q_int.mean():.4f} m³/s")
print(f"Max |Q - Q_analytical|= {np.abs(Q_int - Q_analytical).max() * 1000:.4f} L/s")

## Interpretation of Results

1. **Depth profile:** The simulated depth is close to the analytical h_n = 0.400 m. Any residual offset reflects the subcritical inlet over-constraint: for Fr < 1, fixing both h and u at the boundary imposes one condition more than the hyperbolic system admits, causing a small flux imbalance that propagates into the interior. This is quantified by the MAE and effective-n values above.

2. **Discharge conservation:** Q(x) is uniform along the channel at steady state, confirming mass conservation. The absolute discharge error shows whether the over-constrained inlet injects extra or reduced flux.

3. **Comparison with `RiverFlowDynamics`:** The semi-implicit solver handles subcritical inlets more naturally because its global pressure-correction solve implicitly adjusts the free-surface to be consistent with the Dirichlet BC. For this mild-slope subcritical case, `RiverFlowDynamics` therefore converges more closely to the analytical answer.

4. **Where HLLC excels:** For **supercritical flow**, **hydraulic jumps**, **dry-bed wetting fronts**, and **rapidly varying flows**, HLLC's explicit Riemann solver correctly captures the physics that the semi-implicit scheme cannot. Tutorial 1 demonstrates one such case (steep slope, Fr_n = 1.85).

### Click here for more <a href="https://landlab.csdms.io/tutorials/">Landlab tutorials</a>